<a href="https://colab.research.google.com/github/rperezen/intro-to-ML/blob/main/hw_5_pt_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.optim as optim

# ==========================================
# 2.a Preprocessing & Setup
# ==========================================
# Load dataset
df = pd.read_csv('Housing.csv')

# Select the 5 specified features and the target
features = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
X = df[features].values
y = df['price'].values.reshape(-1, 1)

# Standardize features (crucial to prevent exploding gradients)
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0)
X_scaled = (X - X_mean) / X_std

# Convert to PyTorch Tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# 80/20 Train-Validation Split (Sequential split to match HW2)
split_idx = int(0.8 * len(X))
X_train, y_train = X_tensor[:split_idx], y_tensor[:split_idx]
X_val, y_val = X_tensor[split_idx:], y_tensor[split_idx:]

# Define linear model and standard MSE loss
def model(X, w, b):
    return torch.matmul(X, w) + b

def loss_fn(preds, targets):
    return torch.mean((preds - targets) ** 2)

# ==========================================
# 2.b Training Loop & Hyperparameter Tuning
# ==========================================
epochs = 5000
learning_rates = [0.1, 0.01, 0.001, 0.0001]

best_val_loss = float('inf')
best_params = {}

for lr in learning_rates:
    print(f"\n--- Training with Learning Rate: {lr} ---")

    # Initialize 6 parameters: 5 weights (w) and 1 bias (b)
    w = torch.zeros((5, 1), dtype=torch.float32, requires_grad=True)
    b = torch.zeros(1, dtype=torch.float32, requires_grad=True)

    # Using PyTorch's Stochastic Gradient Descent
    optimizer = optim.SGD([w, b], lr=lr)

    for epoch in range(1, epochs + 1):
        # Forward pass & calculate loss
        preds = model(X_train, w, b)
        train_loss = loss_fn(preds, y_train)

        # Backward pass & optimization
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        # Evaluate and report every 500 epochs with 4 significant figures (.3e)
        if epoch % 500 == 0:
            with torch.no_grad():
                val_preds = model(X_val, w, b)
                val_loss = loss_fn(val_preds, y_val)

            print(f"Epoch {epoch} | Train Loss: {train_loss.item():.3e} | Val Loss: {val_loss.item():.3e}")

    # Keep track of the best final model
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_params = {'lr': lr, 'w': w.detach().clone(), 'b': b.detach().clone()}

# Output best model parameters with 4 significant figures
print("\n==========================================")
print(" BEST MODEL PARAMETERS")
print("==========================================")
print(f"Optimal Learning Rate: {best_params['lr']}")
print(f"Lowest Final Validation Loss: {best_val_loss:.3e}")
print("\nThe Six Identified Parameters:")
for i, feature in enumerate(features):
    print(f" - Weight ({feature}): {best_params['w'][i].item():.2f}")
print(f" - Bias (Intercept): {best_params['b'].item():.2f}")


--- Training with Learning Rate: 0.1 ---
Epoch 500 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 1000 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 1500 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 2000 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 2500 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 3000 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 3500 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 4000 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 4500 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 5000 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12

--- Training with Learning Rate: 0.01 ---
Epoch 500 | Train Loss: 1.499e+12 | Val Loss: 2.132e+12
Epoch 1000 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 1500 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 2000 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 2500 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch 3000 | Train Loss: 1.499e+12 | Val Loss: 2.134e+12
Epoch